# Fabric Tenant-Wide Dataset Refresh Audit

This notebook scans all workspaces in your Microsoft Fabric tenant and retrieves the refresh configuration (Full / Incremental / DirectQuery) for every dataset (semantic model).

It is designed to run **directly inside a Microsoft Fabric notebook**.

## Prerequisites
- Run this notebook with an identity that has **Power BI / Fabric Admin** rights (Fabric Administrator role, or a service principal with `Tenant.Read.All` Admin API access).
- Attaching a **Lakehouse** is optional but recommended if you want to persist the results as a Delta table.
- The default authentication uses the notebook's own identity via `notebookutils.credentials.getToken`, so no secrets are required. A service principal fallback is included for automated jobs.

## 0. Parameters

This cell is tagged as **parameters** so it can be overridden when the notebook is run from a Fabric pipeline or scheduled job.

In [8]:
# Authentication method: 'fabric_integrated' (default, uses notebook identity),
# 'service_principal', or 'default_azure'
auth_method = "fabric_integrated"

# Only required when auth_method == 'service_principal'
tenant_id = ""
client_id = ""
client_secret = ""

# Number of parallel workers for enriching dataset details (mind API rate limits)
max_workers = 5

# Set to True to save results to an attached Lakehouse as a Delta table
save_to_lakehouse = True
lakehouse_table_name = "dataset_refresh_audit"

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 10, Finished, Available, Finished, False)

## 1. Imports

In [9]:
import os
import time
import json
import logging
import requests
import pandas as pd
from typing import Dict, List, Optional
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 11, Finished, Available, Finished, False)

## 2. Authentication

In Fabric, the simplest and most secure option is `fabric_integrated`, which uses the notebook's own identity via `notebookutils.credentials.getToken`. No secrets are stored in the notebook.

For automated (non-interactive) runs outside Fabric, a **service principal** or **DefaultAzureCredential** can be used instead.

In [10]:
class FabricAuthenticator:
    """Handles authentication to the Power BI Admin API."""

    # Audience for Fabric / Power BI REST APIs
    PBI_RESOURCE = "https://analysis.windows.net/powerbi/api"
    PBI_SCOPE = "https://analysis.windows.net/powerbi/api/.default"

    def __init__(
        self,
        tenant_id: Optional[str] = None,
        client_id: Optional[str] = None,
        client_secret: Optional[str] = None,
        auth_method: str = "fabric_integrated",
    ):
        self.tenant_id = tenant_id or os.getenv('AZURE_TENANT_ID')
        self.client_id = client_id or os.getenv('AZURE_CLIENT_ID')
        self.client_secret = client_secret or os.getenv('AZURE_CLIENT_SECRET')
        self.auth_method = auth_method

    def get_access_token(self) -> str:
        """Get an access token for the Power BI Admin API."""
        try:
            if self.auth_method == "fabric_integrated":
                # Uses the notebook's own identity - no secrets required.
                import notebookutils  # available in the Fabric runtime
                token = notebookutils.credentials.getToken(self.PBI_RESOURCE)
                logger.info("Authenticated using Fabric notebook identity")
                return token

            if self.auth_method == "service_principal":
                from azure.identity import ClientSecretCredential
                if not all([self.tenant_id, self.client_id, self.client_secret]):
                    raise ValueError(
                        "Service Principal auth requires tenant_id, client_id and client_secret "
                        "(parameters or AZURE_TENANT_ID / AZURE_CLIENT_ID / AZURE_CLIENT_SECRET env vars)."
                    )
                credential = ClientSecretCredential(
                    tenant_id=self.tenant_id,
                    client_id=self.client_id,
                    client_secret=self.client_secret,
                )
                logger.info("Authenticated using service principal")
                return credential.get_token(self.PBI_SCOPE).token

            # DefaultAzureCredential (CLI, managed identity, env, etc.)
            from azure.identity import DefaultAzureCredential
            credential = DefaultAzureCredential()
            logger.info("Authenticated using DefaultAzureCredential")
            return credential.get_token(self.PBI_SCOPE).token

        except Exception as e:
            logger.error(f"Authentication failed: {e}")
            raise

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 12, Finished, Available, Finished, False)

## 3. Power BI Admin API Client

Handles all interactions with the Power BI API endpoints, including pagination, retries and rate-limit handling.

| Endpoint | Method | Purpose |
|----------|--------|---------|
| `/admin/workspaces` (`/admin/groups`) | GET | List all workspaces in tenant |
| `/admin/datasets` | GET | List all datasets across tenant (returns `targetStorageMode`, `isRefreshable`) |
| `/groups/{groupId}/datasets/{datasetId}/refreshSchedule` | GET | Get the refresh schedule (enabled / days / times) for a dataset |

> **Note on APIs:** There is **no** `GET /admin/datasets/{id}` endpoint, and the `GetDatasetsAsAdmin` response does **not** include `refreshSchedule` or `defaultMode`. Storage mode is derived from `targetStorageMode` / `isRefreshable` (both returned by the list API), and the actual refresh schedule is read from the non-admin **Get Refresh Schedule In Group** endpoint. That in-group call requires the caller to have access to the workspace, so it is treated as best-effort (401/403/404 are handled gracefully).


In [ ]:
class PowerBIAdminAPIClient:
    """Client for interacting with the Power BI Admin API."""

    ADMIN_BASE_URL = "https://api.powerbi.com/v1.0/myorg/admin"
    # Refresh schedule is only exposed via the non-admin (in-group) endpoint.
    BASE_URL = "https://api.powerbi.com/v1.0/myorg"

    def __init__(self, authenticator: FabricAuthenticator):
        self.authenticator = authenticator
        self.session = requests.Session()
        self.headers = {}
        self._refresh_headers()

    def _refresh_headers(self):
        token = self.authenticator.get_access_token()
        self.headers = {
            'Authorization': f'Bearer {token}',
            'Content-Type': 'application/json',
        }

    def _make_request(
        self,
        method: str,
        endpoint: str,
        params: Optional[Dict] = None,
        data: Optional[Dict] = None,
        retry_count: int = 3,
        base_url: Optional[str] = None,
    ) -> Dict:
        url = f"{base_url or self.ADMIN_BASE_URL}{endpoint}"

        for attempt in range(retry_count):
            try:
                response = self.session.request(
                    method=method,
                    url=url,
                    headers=self.headers,
                    params=params,
                    json=data,
                    timeout=30,
                )

                # Handle rate limiting
                if response.status_code == 429:
                    retry_after = int(response.headers.get('Retry-After', 60))
                    logger.warning(f"Rate limited. Waiting {retry_after}s...")
                    time.sleep(retry_after)
                    continue

                response.raise_for_status()
                return response.json() if response.text else {}

            except requests.exceptions.RequestException as e:
                if attempt < retry_count - 1:
                    wait_time = 2 ** attempt  # Exponential backoff
                    logger.warning(
                        f"Request failed (attempt {attempt + 1}/{retry_count}): {e}. "
                        f"Retrying in {wait_time}s..."
                    )
                    time.sleep(wait_time)
                else:
                    logger.error(f"Request failed after {retry_count} attempts: {e}")
                    raise
        return {}

    def get_workspaces(self) -> List[Dict]:
        """Get all workspaces in the tenant (paginated).

        Uses the ``GET /admin/groups`` (GetGroupsAsAdmin) endpoint. Note there
        is no ``/admin/workspaces`` endpoint in the v1.0 Admin API.
        """
        logger.info("Fetching all workspaces (via /admin/groups)...")
        all_workspaces, skip, top = [], 0, 100
        while True:
            response = self._make_request(
                method='GET', endpoint='/groups',
                params={'$skip': skip, '$top': top},
            )
            workspaces = response.get('value', [])
            if not workspaces:
                break
            all_workspaces.extend(workspaces)
            logger.info(f"Fetched {len(workspaces)} workspaces (total: {len(all_workspaces)})")
            skip += top
        logger.info(f"Total workspaces found: {len(all_workspaces)}")
        return all_workspaces

    def get_datasets(self) -> List[Dict]:
        """Get all datasets across the entire tenant (paginated).

        The GetDatasetsAsAdmin response already includes the storage-mode
        signals we need (``targetStorageMode`` and ``isRefreshable``), so no
        per-dataset admin enrichment call is required (there is no
        ``GET /admin/datasets/{id}`` endpoint).
        """
        logger.info("Fetching all datasets across tenant...")
        all_datasets, skip, top = [], 0, 100
        while True:
            response = self._make_request(
                method='GET', endpoint='/datasets',
                params={'$skip': skip, '$top': top},
            )
            datasets = response.get('value', [])
            if not datasets:
                break
            all_datasets.extend(datasets)
            logger.info(f"Fetched {len(datasets)} datasets (total: {len(all_datasets)})")
            skip += top
        logger.info(f"Total datasets found: {len(all_datasets)}")
        return all_datasets

    def get_refresh_schedule(self, workspace_id: str, dataset_id: str) -> Dict:
        """Get the refresh schedule for a dataset via the in-group endpoint.

        Uses ``GET /groups/{groupId}/datasets/{datasetId}/refreshSchedule``.
        There is no Admin variant of this call, so the caller must have access
        to the workspace. Access/availability failures (401/403/404) are
        returned as an empty dict so the audit can continue.

        Returns a dict like ``{'days': [...], 'times': [...], 'enabled': bool,
        'localTimeZoneId': str, 'notifyOption': str}`` when available.
        """
        try:
            return self._make_request(
                method='GET',
                endpoint=f'/groups/{workspace_id}/datasets/{dataset_id}/refreshSchedule',
                base_url=self.BASE_URL,
                retry_count=1,
            )
        except requests.exceptions.HTTPError as e:
            status = getattr(e.response, 'status_code', None)
            if status in (401, 403, 404):
                # DirectQuery/Live datasets have no schedule (404); admin may
                # lack workspace access (401/403). Not an error for the audit.
                logger.debug(
                    f"No accessible refresh schedule for dataset {dataset_id} "
                    f"in workspace {workspace_id} (HTTP {status})."
                )
            else:
                logger.warning(
                    f"Error fetching refresh schedule for dataset {dataset_id}: {e}"
                )
            return {}
        except Exception as e:
            logger.warning(f"Error fetching refresh schedule for dataset {dataset_id}: {e}")
            return {}


StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 13, Finished, Available, Finished, False)

## 4. Refresh Configuration Parser

Interprets the refresh configuration by combining the storage-mode signals from `GetDatasetsAsAdmin` (`targetStorageMode`, `isRefreshable`) with the refresh schedule fetched from the in-group *Get Refresh Schedule* endpoint.

| Refresh Type | Meaning | Signals |
|--------------|---------|---------|
| Scheduled Refresh | Import model with an enabled refresh schedule | `refreshSchedule.enabled = true` |
| Schedule Disabled | Import model with a schedule that is turned off | `refreshSchedule.enabled = false` |
| Refreshable (No Schedule) | Import model, refreshable, no schedule found | `isRefreshable = true`, no schedule |
| DirectQuery/Live (No Refresh) | Live connection, no scheduled refresh | `targetStorageMode = DirectQuery` or `isRefreshable = false` |
| Not Configured | No refresh signals available | none |

> **Incremental refresh** is *not* returned by these APIs. Detecting incremental vs. full requires the metadata scanner API (model definition/expressions), so `Is Incremental` is reported as *Unknown (not exposed by Admin API)*.


In [ ]:
class RefreshConfigParser:
    """Parse and interpret refresh configurations from dataset metadata.

    Storage mode comes from the GetDatasetsAsAdmin response
    (``targetStorageMode`` / ``isRefreshable``). The refresh schedule
    (``enabled`` / ``days`` / ``times``) is fetched separately from the
    in-group *Get Refresh Schedule* endpoint and attached under the
    ``refreshSchedule`` key before this parser runs.

    Note: Incremental vs. full refresh is **not** exposed by these APIs
    (the incremental policy lives in the model definition and is only
    available via the metadata scanner API), so it is reported as unknown.
    """

    @staticmethod
    def extract_refresh_config(dataset: Dict) -> Dict:
        refresh_config = {
            'refresh_type': 'Not Configured',
            'is_incremental': 'Unknown (not exposed by Admin API)',
            'refresh_schedule_enabled': False,
            'scheduled_refresh_times': [],
            'raw_config': {},
        }

        # Storage mode from the admin list response.
        storage_mode = (dataset.get('targetStorageMode') or '').strip()
        is_refreshable = dataset.get('isRefreshable')

        # Refresh schedule fetched from the in-group refreshSchedule endpoint.
        schedule = dataset.get('refreshSchedule') or {}
        schedule_enabled = bool(schedule.get('enabled'))
        refresh_config['refresh_schedule_enabled'] = schedule_enabled
        refresh_config['scheduled_refresh_times'] = schedule.get('times', []) or []
        refresh_config['raw_config'] = schedule

        # DirectQuery / LiveConnection models are not refreshable.
        if storage_mode == 'DirectQuery' or is_refreshable is False:
            refresh_config['refresh_type'] = 'DirectQuery/Live (No Refresh)'
        else:
            # Import (or refreshable) model.
            if schedule_enabled:
                refresh_config['refresh_type'] = 'Scheduled Refresh'
            elif schedule:
                # A schedule object exists but is disabled.
                refresh_config['refresh_type'] = 'Schedule Disabled'
            elif is_refreshable:
                refresh_config['refresh_type'] = 'Refreshable (No Schedule)'
            else:
                refresh_config['refresh_type'] = 'Not Configured'

        return refresh_config


StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 14, Finished, Available, Finished, False)

## 5. Tenant Audit Engine

Orchestrates scanning of all workspaces and datasets, then builds a structured results table.

In [ ]:
class TenantDatasetAudit:
    """Main audit engine for tenant-wide dataset discovery and refresh analysis."""

    def __init__(self, client: PowerBIAdminAPIClient):
        self.client = client
        self.workspaces = {}
        self.datasets = []

    def run_audit(self, max_workers: int = 5) -> pd.DataFrame:
        logger.info("Starting tenant-wide dataset audit...")
        start_time = datetime.now()

        self._fetch_workspaces()
        self._fetch_all_datasets()
        self._enrich_datasets(max_workers=max_workers)
        results_df = self._build_results_dataframe()

        elapsed = (datetime.now() - start_time).total_seconds()
        logger.info(
            f"Audit completed in {elapsed:.2f}s. "
            f"Found {len(self.workspaces)} workspaces, {len(results_df)} datasets."
        )
        return results_df

    def _fetch_workspaces(self):
        logger.info("Step 1: Fetching workspaces...")
        for workspace in self.client.get_workspaces():
            self.workspaces[workspace['id']] = {
                'id': workspace['id'],
                'name': workspace.get('name', 'Unknown'),
                'type': workspace.get('type', 'Unknown'),
                'state': workspace.get('state', 'Unknown'),
            }
        logger.info(f"Cached {len(self.workspaces)} workspaces")

    def _fetch_all_datasets(self):
        logger.info("Step 2: Fetching all datasets...")
        self.datasets = self.client.get_datasets()
        logger.info(f"Found {len(self.datasets)} datasets")

    def _enrich_datasets(self, max_workers: int = 5):
        """Attach each dataset's refresh schedule (best-effort).

        The GetDatasetsAsAdmin list already provides storage-mode signals, so
        enrichment only fetches the refresh schedule from the in-group
        endpoint. Datasets without a workspace ID, or in workspaces the caller
        can't access, are skipped gracefully.
        """
        logger.info(f"Step 3: Fetching refresh schedules (max_workers={max_workers})...")

        def _fetch(ds):
            workspace_id = ds.get('workspaceId')
            if not workspace_id:
                return {}
            return self.client.get_refresh_schedule(workspace_id, ds['id'])

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(_fetch, ds): ds for ds in self.datasets}
            completed = 0
            for future in as_completed(futures):
                try:
                    schedule = future.result()
                    if schedule:
                        futures[future]['refreshSchedule'] = schedule
                    completed += 1
                    if completed % 100 == 0:
                        logger.info(f"Processed {completed}/{len(self.datasets)} datasets")
                except Exception as e:
                    logger.warning(f"Failed to fetch refresh schedule: {e}")
        logger.info(f"Completed refresh-schedule enrichment of {len(self.datasets)} datasets")

    def _build_results_dataframe(self) -> pd.DataFrame:
        logger.info("Building results dataframe...")
        results = []
        for dataset in self.datasets:
            workspace_id = dataset.get('workspaceId', '')
            workspace = self.workspaces.get(workspace_id, {})
            refresh_config = RefreshConfigParser.extract_refresh_config(dataset)
            results.append({
                'Workspace Name': workspace.get('name', 'Unknown'),
                'Workspace ID': workspace_id,
                'Dataset Name': dataset.get('name', 'Unknown'),
                'Dataset ID': dataset.get('id', ''),
                'Refresh Type': refresh_config['refresh_type'],
                'Is Incremental': refresh_config['is_incremental'],
                'Schedule Enabled': refresh_config['refresh_schedule_enabled'],
                'Scheduled Times': ', '.join(refresh_config['scheduled_refresh_times'])
                    if refresh_config['scheduled_refresh_times'] else 'None',
                'Is Refreshable': dataset.get('isRefreshable', 'Unknown'),
                'Owner': dataset.get('configuredBy', dataset.get('owner', 'Unknown')),
                'Target Storage Mode': dataset.get('targetStorageMode', 'Unknown'),
            })
        df = pd.DataFrame(results)
        if not df.empty:
            df = df.sort_values(['Workspace Name', 'Dataset Name']).reset_index(drop=True)
        logger.info(f"Results dataframe built: {len(df)} rows")
        return df


StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 15, Finished, Available, Finished, False)

## 6. Run the Audit

This uses the parameters defined at the top of the notebook.

In [ ]:
logger.info("=" * 80)
logger.info("FABRIC TENANT-WIDE DATASET REFRESH AUDIT")
logger.info("=" * 80)

authenticator = FabricAuthenticator(
    tenant_id=tenant_id or None,
    client_id=client_id or None,
    client_secret=client_secret or None,
    auth_method=auth_method,
)

client = PowerBIAdminAPIClient(authenticator)

audit = TenantDatasetAudit(client)
results_df = audit.run_audit(max_workers=max_workers)

print(f"\nTotal Datasets:   {len(results_df)}")
if not results_df.empty:
    print(f"Total Workspaces: {results_df['Workspace ID'].nunique()}")
    print("\nRefresh Type Distribution:")
    print(results_df['Refresh Type'].value_counts())
    print("\nStorage Mode Distribution:")
    print(results_df['Target Storage Mode'].value_counts())
    print(f"\nSchedule Enabled: {results_df['Schedule Enabled'].sum()} datasets")


StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 16, Finished, Available, Finished, False)

2026-07-02 02:19:55,590 - INFO - ================================================================================
2026-07-02 02:19:55,591 - INFO - FABRIC TENANT-WIDE DATASET REFRESH AUDIT
2026-07-02 02:19:55,592 - INFO - ================================================================================
2026-07-02 02:19:55,597 - INFO - Authenticated using Fabric notebook identity
2026-07-02 02:19:55,598 - INFO - Starting tenant-wide dataset audit...
2026-07-02 02:19:55,598 - INFO - Step 1: Fetching workspaces...
2026-07-02 02:19:55,599 - INFO - Fetching all workspaces (via /admin/groups)...
2026-07-02 02:19:55,888 - INFO - Fetched 65 workspaces (total: 65)
2026-07-02 02:19:55,961 - INFO - Total workspaces found: 65
2026-07-02 02:19:55,962 - INFO - Cached 65 workspaces
2026-07-02 02:19:55,962 - INFO - Step 2: Fetching all datasets...
2026-07-02 02:19:55,963 - INFO - Fetching all datasets across tenant...
2026-07-02 02:19:56,138 - INFO - Fetched 66 datasets (total: 66)
2026-07-02 02:19:56,2


Total Datasets:   66
Total Workspaces: 29

Refresh Type Distribution:
Refresh Type
Not Configured    66
Name: count, dtype: int64

Dataset Mode Distribution:
Default Mode
Unknown    66
Name: count, dtype: int64

Schedule Enabled: 0 datasets


In [15]:
# Display the full results table (rich table view in Fabric)
display(results_df)

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 81d986ea-7618-44f4-b1ad-da675e26cdfd)

## 7. Save Results to Lakehouse (optional)

Set `save_to_lakehouse = True` in the parameters cell and attach a Lakehouse to persist the audit as a Delta table.

In [17]:
if save_to_lakehouse and not results_df.empty:
    import re

    # Add an audit timestamp column
    export_df = results_df.copy()
    export_df['Audit Timestamp'] = datetime.utcnow().isoformat()

    # Sanitize column names for Delta: replace spaces and other invalid chars with '_'
    invalid_chars_pattern = r"[ ,;{}()\n\t=]"
    export_df.columns = [re.sub(invalid_chars_pattern, "_", str(c)) for c in export_df.columns]

    # Use the Fabric-provided Spark session (do NOT create a new one)
    spark_df = spark.createDataFrame(export_df)
    (spark_df.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(lakehouse_table_name))

    print(f"✓ Results saved to Lakehouse table: {lakehouse_table_name}")
else:
    print("Skipping Lakehouse save (save_to_lakehouse is False or no results).")

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 19, Finished, Available, Finished, False)

✓ Results saved to Lakehouse table: dataset_refresh_audit


In [19]:
df = spark.sql("SELECT * FROM Fabric_Audit_LH.dataset_refresh_audit LIMIT 1000")
display(df)

StatementMeta(, 0e18a0f3-de53-4bb5-b985-fecdd00934d6, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ac6636db-5bd1-45e5-baa6-b63c4b1f955f)